# Read-Level Benchmarking for Baleen

This notebook evaluates the pipeline's ability to correctly classify **individual reads** as modified or unmodified.

## Metrics
- **ROC-AUC**: Overall discriminative ability
- **Precision-Recall AUC**: Performance on the modified (positive) class
- **Calibration**: How well probabilities reflect true modification rates
- **Stratified analysis**: Performance by coverage, effect size, and read count

In [ ]:
import sys
from pathlib import Path

# Add parent to path if running from notebooks directory
if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.calibration import calibration_curve
import pandas as pd
import seaborn as sns

from baleen.eventalign._hierarchical import (
    compute_sequential_modification_probabilities,
    _extract_ivt_distances,
)
from baleen.eventalign._probability import (
    knn_ivt_purity,
    distance_to_ivt,
    mds_gmm,
)
from baleen.eventalign._pipeline import PositionResult, ContigResult

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Synthetic Data Generation

Create distance matrices with known modification states.

In [ ]:
def make_block_distance_matrix(
    n_native: int,
    n_ivt: int,
    within_native: float = 1.0,
    within_ivt: float = 1.0,
    between: float = 5.0,
    noise: float = 0.1,
    seed: int = 42,
) -> np.ndarray:
    """Create a distance matrix with block structure.
    
    Modified positions: native-native distances are small, native-IVT are large.
    Unmodified positions: all distances are similar.
    """
    rng = np.random.RandomState(seed)
    n = n_native + n_ivt
    mat = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            if i < n_native and j < n_native:
                d = within_native
            elif i >= n_native and j >= n_native:
                d = within_ivt
            else:
                d = between
            d += rng.normal(0, noise)
            d = max(d, 0.0)
            mat[i, j] = d
            mat[j, i] = d
    return mat


def make_homogeneous_matrix(
    n_native: int,
    n_ivt: int,
    base_dist: float = 1.0,
    noise: float = 0.05,
    seed: int = 42,
) -> np.ndarray:
    """Create a homogeneous distance matrix (unmodified position)."""
    rng = np.random.RandomState(seed)
    n = n_native + n_ivt
    mat = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = base_dist + rng.normal(0, noise)
            d = max(d, 0.0)
            mat[i, j] = d
            mat[j, i] = d
    return mat

## 2. Single Position Benchmark

Evaluate classification performance for a single position with varying effect sizes.

In [ ]:
def benchmark_single_position(
    n_native: int = 20,
    n_ivt: int = 15,
    effect_size: float = 5.0,  # between - within_ivt
    noise: float = 0.1,
    seed: int = 42,
):
    """Benchmark a single modified position.
    
    Returns dict with algorithm results and ground truth.
    """
    dm = make_block_distance_matrix(
        n_native, n_ivt,
        within_native=1.5, within_ivt=1.0,
        between=effect_size + 1.0,
        noise=noise,
        seed=seed,
    )
    
    # Ground truth: native reads are modified, IVT reads are unmodified
    y_true = np.array([1] * n_native + [0] * n_ivt)
    
    # Run all three algorithms
    results = {}
    
    # kNN IVT-purity (default algorithm)
    knn_result = knn_ivt_purity(dm, n_native, n_ivt)
    results['knn_ivt_purity'] = {
        'probs': knn_result.probabilities,
        'scores': knn_result.scores,
    }
    
    # Distance-to-IVT
    dti_result = distance_to_ivt(dm, n_native, n_ivt)
    results['distance_to_ivt'] = {
        'probs': dti_result.probabilities,
        'scores': dti_result.scores,
    }
    
    # MDS + GMM
    mds_result = mds_gmm(dm, n_native, n_ivt)
    results['mds_gmm'] = {
        'probs': mds_result.probabilities,
        'scores': mds_result.scores,
    }
    
    return {
        'y_true': y_true,
        'results': results,
        'distance_matrix': dm,
    }

In [ ]:
# Run benchmark with default parameters
bench = benchmark_single_position(n_native=30, n_ivt=20, effect_size=5.0, noise=0.15)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (algo_name, algo_data) in zip(axes, bench['results'].items()):
    y_true = bench['y_true']
    probs = algo_data['probs']
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, probs)
    roc_auc = auc(fpr, tpr)
    
    # PR curve
    precision, recall, _ = precision_recall_curve(y_true, probs)
    pr_auc = average_precision_score(y_true, probs)
    
    # Plot ROC
    ax.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{algo_name}\nPR-AUC={pr_auc:.3f}')
    ax.legend(loc='lower right')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('Read-Level Classification (Single Modified Position)', y=1.02)
plt.tight_layout()
plt.savefig('read_level_single_position_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Effect Size Sweep

How does performance vary with signal strength (native-IVT separation)?

In [ ]:
effect_sizes = np.linspace(1.0, 10.0, 10)
n_trials = 5

results_by_effect = {
    'knn_ivt_purity': {'roc_auc': [], 'pr_auc': []},
    'distance_to_ivt': {'roc_auc': [], 'pr_auc': []},
    'mds_gmm': {'roc_auc': [], 'pr_auc': []},
}

for effect in effect_sizes:
    trial_roc = {k: [] for k in results_by_effect}
    trial_pr = {k: [] for k in results_by_effect}
    
    for trial in range(n_trials):
        bench = benchmark_single_position(
            n_native=25, n_ivt=15,
            effect_size=effect,
            noise=0.15,
            seed=trial * 100 + int(effect * 10),
        )
        y_true = bench['y_true']
        
        for algo_name, algo_data in bench['results'].items():
            probs = algo_data['probs']
            if np.all(probs == 0):
                # Null gate fired, skip this trial
                continue
            fpr, tpr, _ = roc_curve(y_true, probs)
            trial_roc[algo_name].append(auc(fpr, tpr))
            trial_pr[algo_name].append(average_precision_score(y_true, probs))
    
    for algo in results_by_effect:
        results_by_effect[algo]['roc_auc'].append(np.mean(trial_roc[algo]) if trial_roc[algo] else 0)
        results_by_effect[algo]['pr_auc'].append(np.mean(trial_pr[algo]) if trial_pr[algo] else 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'knn_ivt_purity': '#2196F3', 'distance_to_ivt': '#4CAF50', 'mds_gmm': '#FF9800'}
labels = {'knn_ivt_purity': 'kNN IVT-Purity (default)', 
          'distance_to_ivt': 'Distance-to-IVT', 
          'mds_gmm': 'MDS + GMM'}

for algo in results_by_effect:
    axes[0].plot(effect_sizes, results_by_effect[algo]['roc_auc'], 
                 'o-', color=colors[algo], label=labels[algo], lw=2, markersize=6)
    axes[1].plot(effect_sizes, results_by_effect[algo]['pr_auc'],
                 'o-', color=colors[algo], label=labels[algo], lw=2, markersize=6)

axes[0].set_xlabel('Effect Size (Native-IVT Separation)')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('ROC-AUC vs Effect Size')
axes[0].legend()
axes[0].set_ylim([0.4, 1.05])

axes[1].set_xlabel('Effect Size (Native-IVT Separation)')
axes[1].set_ylabel('PR-AUC')
axes[1].set_title('PR-AUC vs Effect Size')
axes[1].legend()
axes[1].set_ylim([0.4, 1.05])

plt.suptitle('Read-Level Performance vs Signal Strength', y=1.02)
plt.tight_layout()
plt.savefig('read_level_effect_size_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Coverage Sweep

How does performance vary with IVT coverage?

In [ ]:
ivt_coverages = [3, 5, 10, 15, 20, 30, 50]
n_native = 25
effect_size = 5.0
n_trials = 5

results_by_coverage = {
    'knn_ivt_purity': {'roc_auc': [], 'pr_auc': []},
    'distance_to_ivt': {'roc_auc': [], 'pr_auc': []},
    'mds_gmm': {'roc_auc': [], 'pr_auc': []},
}

for n_ivt in ivt_coverages:
    trial_roc = {k: [] for k in results_by_coverage}
    trial_pr = {k: [] for k in results_by_coverage}
    
    for trial in range(n_trials):
        bench = benchmark_single_position(
            n_native=n_native, n_ivt=n_ivt,
            effect_size=effect_size,
            noise=0.15,
            seed=trial * 1000 + n_ivt,
        )
        y_true = bench['y_true']
        
        for algo_name, algo_data in bench['results'].items():
            probs = algo_data['probs']
            if np.all(probs == 0):
                continue
            fpr, tpr, _ = roc_curve(y_true, probs)
            trial_roc[algo_name].append(auc(fpr, tpr))
            trial_pr[algo_name].append(average_precision_score(y_true, probs))
    
    for algo in results_by_coverage:
        results_by_coverage[algo]['roc_auc'].append(np.mean(trial_roc[algo]) if trial_roc[algo] else 0)
        results_by_coverage[algo]['pr_auc'].append(np.mean(trial_pr[algo]) if trial_pr[algo] else 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for algo in results_by_coverage:
    axes[0].plot(ivt_coverages, results_by_coverage[algo]['roc_auc'],
                 'o-', color=colors[algo], label=labels[algo], lw=2, markersize=8)
    axes[1].plot(ivt_coverages, results_by_coverage[algo]['pr_auc'],
                 'o-', color=colors[algo], label=labels[algo], lw=2, markersize=8)

axes[0].set_xlabel('IVT Coverage (number of reads)')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('ROC-AUC vs IVT Coverage')
axes[0].legend()
axes[0].set_ylim([0.4, 1.05])

axes[1].set_xlabel('IVT Coverage (number of reads)')
axes[1].set_ylabel('PR-AUC')
axes[1].set_title('PR-AUC vs IVT Coverage')
axes[1].legend()
axes[1].set_ylim([0.4, 1.05])

# Add coverage class annotations
for ax in axes:
    ax.axvline(x=5, color='gray', ls=':', alpha=0.5)
    ax.axvline(x=20, color='gray', ls=':', alpha=0.5)
    ax.text(3.5, ax.get_ylim()[1] - 0.1, 'LOW', fontsize=9, color='gray')
    ax.text(11, ax.get_ylim()[1] - 0.1, 'MEDIUM', fontsize=9, color='gray')
    ax.text(32, ax.get_ylim()[1] - 0.1, 'HIGH', fontsize=9, color='gray')

plt.suptitle('Read-Level Performance vs IVT Coverage (Effect Size=5.0)', y=1.02)
plt.tight_layout()
plt.savefig('read_level_coverage_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Probability Calibration

Check if predicted probabilities reflect true modification rates.

In [ ]:
# Aggregate predictions across many trials for calibration analysis
all_probs = {'knn_ivt_purity': [], 'distance_to_ivt': [], 'mds_gmm': []}
all_labels = []

for trial in range(50):
    bench = benchmark_single_position(
        n_native=20, n_ivt=15,
        effect_size=5.0,
        noise=0.15,
        seed=trial * 7 + 13,
    )
    all_labels.extend(bench['y_true'])
    for algo in all_probs:
        all_probs[algo].extend(bench['results'][algo]['probs'])

all_labels = np.array(all_labels)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (algo_name, probs_list) in zip(axes, all_probs.items()):
    probs = np.array(probs_list)
    
    # Calibration curve
    prob_true, prob_pred = calibration_curve(all_labels, probs, n_bins=10, strategy='uniform')
    
    ax.plot(prob_pred, prob_true, 's-', color=colors[algo_name], lw=2, label=algo_name)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectly calibrated')
    
    # Histogram of predictions
    ax2 = ax.twinx()
    ax2.hist(probs, bins=20, alpha=0.3, color='gray', density=True)
    ax2.set_ylabel('Density', color='gray')
    
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'{labels[algo_name]}')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('Probability Calibration Curves', y=1.02)
plt.tight_layout()
plt.savefig('read_level_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Score Distribution Visualization

In [ ]:
# Compare score distributions for modified vs unmodified reads
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (algo_name, probs_list) in zip(axes, all_probs.items()):
    probs = np.array(probs_list)
    
    # Separate by true label
    modified_probs = probs[all_labels == 1]
    unmodified_probs = probs[all_labels == 0]
    
    ax.hist(unmodified_probs, bins=30, alpha=0.6, label='Unmodified (IVT)', 
            color='#FF9800', density=True)
    ax.hist(modified_probs, bins=30, alpha=0.6, label='Modified (Native)',
            color='#2196F3', density=True)
    
    ax.set_xlabel('P(modified)')
    ax.set_ylabel('Density')
    ax.set_title(f'{labels[algo_name]}')
    ax.legend()
    ax.set_xlim([-0.02, 1.02])

plt.suptitle('Score Distribution by True Modification Status', y=1.02)
plt.tight_layout()
plt.savefig('read_level_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Noise Robustness

In [ ]:
noise_levels = np.linspace(0.05, 0.5, 10)
n_trials = 5

results_by_noise = {
    'knn_ivt_purity': {'roc_auc': []},
    'distance_to_ivt': {'roc_auc': []},
    'mds_gmm': {'roc_auc': []},
}

for noise in noise_levels:
    trial_roc = {k: [] for k in results_by_noise}
    
    for trial in range(n_trials):
        bench = benchmark_single_position(
            n_native=25, n_ivt=15,
            effect_size=5.0,
            noise=noise,
            seed=trial * 100 + int(noise * 100),
        )
        y_true = bench['y_true']
        
        for algo_name, algo_data in bench['results'].items():
            probs = algo_data['probs']
            if np.all(probs == 0):
                continue
            fpr, tpr, _ = roc_curve(y_true, probs)
            trial_roc[algo_name].append(auc(fpr, tpr))
    
    for algo in results_by_noise:
        results_by_noise[algo]['roc_auc'].append(np.mean(trial_roc[algo]) if trial_roc[algo] else 0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for algo in results_by_noise:
    ax.plot(noise_levels, results_by_noise[algo]['roc_auc'],
            'o-', color=colors[algo], label=labels[algo], lw=2, markersize=8)

ax.set_xlabel('Noise Level (σ)')
ax.set_ylabel('ROC-AUC')
ax.set_title('Read-Level Performance vs Noise Level')
ax.legend()
ax.set_ylim([0.4, 1.05])

plt.tight_layout()
plt.savefig('read_level_noise_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary Table

In [ ]:
# Final summary at default conditions
summary_data = []
for trial in range(20):
    bench = benchmark_single_position(
        n_native=25, n_ivt=15, effect_size=5.0, noise=0.15, seed=trial * 11
    )
    y_true = bench['y_true']
    
    for algo_name, algo_data in bench['results'].items():
        probs = algo_data['probs']
        if np.all(probs == 0):
            continue
        
        fpr, tpr, _ = roc_curve(y_true, probs)
        roc_auc = auc(fpr, tpr)
        pr_auc = average_precision_score(y_true, probs)
        
        # F1 at optimal threshold
        from sklearn.metrics import f1_score
        thresholds = np.linspace(0, 1, 100)
        f1_scores = [f1_score(y_true, probs > t) for t in thresholds]
        best_f1 = max(f1_scores)
        
        summary_data.append({
            'Algorithm': algo_name,
            'ROC-AUC': roc_auc,
            'PR-AUC': pr_auc,
            'Best F1': best_f1,
        })

summary_df = pd.DataFrame(summary_data).groupby('Algorithm').agg(['mean', 'std'])
summary_df.columns = ['_'.join(col) for col in summary_df.columns]
print(summary_df.round(3).to_string())

In [ ]:
summary_df.round(3)